[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/earthquake/earthquake.ipynb)

# Earthquake Beachballs

This example draws earthquake beachballs, the diagrams seismologists use to summarize how a fault slipped. When a fault slips, the ground on one side lurches past the other, and seismometers all around the globe feel the first wave arrive either as a push or as a pull, depending on the direction they lie in. The beachball is the lower half of the sphere of directions, flattened, with the pushes shaded. From it the fault can be recovered, up to one ambiguity that no seismometer can resolve.

A fault's orientation is modelled as one rotor, three turns by its strike, dip and rake, which carries north onto the slip and up onto the fault's normal. The slip acts on the ground as a moment, a map from directions to directions, `slips * normals.dot(Point) + normals * slips.dot(Point)`: the first term alone is a couple, which would turn the ground by the plane of shear `slips.commutator(normals)`, and the second, its mirror, cancels that turning. Along a direction the P wave's first motion is `direction.dot(moment(direction))`, and the fault comes back from the moment's eigenvectors, its pressure and tension axes. The moment is symmetric in the normal and the slip, so the fault and the plane square to its slip give the same moment.

In the tensor notation of seismology the moment reads as $M_{ij} = M_0\,(n_i s_j + n_j s_i)$, and the first motion of the P wave along the unit direction $\gamma$ as $\gamma_i M_{ij} \gamma_j$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_animation
from examples.mechanics.earthquake import render
from examples.mechanics.earthquake.core import directions

ga = VGA3D
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Directions are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
Moment = ga.gatype((Point, Point))                         # Point <- Point

# Space is north, east and down, as in seismology; the planes the fault's frame turns in, from one direction
# towards another, are their commutators.
north, east, down = mv.basis().dual()                      # [] Point each
horizontal = east.commutator(north)                        # [] Bivector: north towards east
upright = down.commutator(east)                            # [] Bivector: east towards down

## 1. The fault as a rotor

A fault is a plane in the ground and a direction of slip within it, given by three angles: the strike, the compass direction of the fault's horizontal line; the dip, how steeply it plunges, to the right of the strike; and the rake, the direction of slip within the plane. Each angle is a turn, and the three compose into one rotor that carries north onto the slip and up onto the fault's normal.

In [ ]:
# The faults: strike-slip, normal, thrust and oblique; their strikes, dips and rakes, in degrees, and the
# seismic moment of each.
strikes = np.array([30.0, 30.0, 30.0, 30.0])
dips = np.array([90.0, 60.0, 30.0, 50.0])
rakes = np.array([0.0, -90.0, 90.0, 45.0])
seismic_moment = 1.0

In [ ]:
def fault_frame(strikes: np.ndarray, dips: np.ndarray, rakes: np.ndarray) -> Rotor:
    """The turn by the rake in the horizontal plane, then the dip about north, then the strike from north."""
    rake = (horizontal * (rakes / 2)).exp()                                   # [...] Rotor: north towards -east
    dip = (upright * (-dips / 2)).exp()                                       # [...] Rotor: east towards down
    strike = (horizontal * (-strikes / 2)).exp()                              # [...] Rotor: north towards east
    return strike * dip * rake


frames = fault_frame(np.deg2rad(strikes), np.deg2rad(dips), np.deg2rad(rakes))   # [faults] Rotor
normals, slips = frames >> -down, frames >> north                              # [faults] Point each

## 2. The moment of a slip

The slip pushes the ground on one side of the fault along the slip and the other side back: a couple, the slip times the normal's component of whatever direction it is given. A couple alone would set the ground turning; how much, its curl, is the couple's commutator with the reciprocal of `Point`, the space of all points, `Point.reverse()`, contracted, and comes out as the plane of shear `slip.commutator(normal)`. The ground does not turn, so a mirrored couple balances it, and the moment is the sum, with no curl. It is symmetric in the normal and the slip: the fault and the plane perpendicular to its slip, the auxiliary plane, have the same moment.

In [ ]:
def double_couple(normals: Point, slips: Point, moments: float) -> Moment:
    """The slip's couple across the fault, and the mirrored couple that keeps the ground from turning."""
    return moments * (slips * normals.dot(Point) + normals * slips.dot(Point))     # [...] Point <- Point


def curl(moment: Moment) -> Bivector:
    """How much a moment would turn the ground."""
    # The reciprocal of a unit antivector is its reverse, and the commutator of two directions the plane they span.
    return Point.reverse().commutator(moment(Point)).contract(1, 2)          # [...] Bivector


# One couple alone, the moment, and the moment with the normal and the slip exchanged.
single = slips * normals.dot(Point)                                           # [faults] Point <- Point
moment = double_couple(normals, slips, seismic_moment)                        # [faults] Point <- Point
auxiliary = double_couple(slips, normals, seismic_moment)                     # [faults] Point <- Point

## 3. First motions and the beachball

Along a direction, the P wave's first motion is the component of the moment's response along that direction: positive for a push away from the source, negative for a pull. The S wave carries the rest of the response, across the direction. Over the lower hemisphere, two turns per direction, the pushes are shaded and the nodal lines where the P wave is silent are the fault and the auxiliary plane. The moment's eigenvectors are its pressure axis, in the middle of the pulls, and its tension axis, in the middle of the pushes.

The balls are the strike-slip, normal, thrust and oblique faults, north up; the pressure axis is the blue dot and the tension axis the red one.

In [ ]:
# The grid of directions: rings from straight down, spokes around the vertical.
n_rings = 60
n_spokes = 120

In [ ]:
def radiation(moment: Moment, directions: Point) -> tuple[Scalar, Point]:
    """The P wave's first motion along each direction, and the S wave's."""
    response = moment(directions)                                             # [...] Point
    push = directions.dot(response)                                           # [...] Scalar
    return push, response - directions * push


hemisphere = directions(np.linspace(0.0, np.pi / 2, n_rings), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [rings, spokes + 1] Point
# The principal values and axes, from pressure through the null axis to tension.
values, axes = moment.eigh()                                                  # [faults, 3] Scalar, Point
push, shear = radiation(moment[:, None, None], hemisphere)                    # [faults, rings, spokes + 1]
save_figure(render.draw_mechanisms(hemisphere, push, axes), "earthquake_beachballs");

## 4. From the moment back to the fault

The moment sends the tension axis to itself and the pressure axis to its opposite, and the normal and slip are their sum and difference, over the square root of two. Which is which, the moment cannot say: both choices rebuild it, and only other evidence, such as the trace of the fault at the surface or the spread of aftershocks, picks the fault from the auxiliary plane.

In [ ]:
def fault_planes(moment: Moment) -> tuple[Point, Point]:
    """The two faults with this moment, each fault's normal the other's slip."""
    _, axes = moment.eigh()                                                   # [..., 3] Point
    pressure, tension = axes[..., 0], axes[..., 2]
    first, second = (tension + pressure) / np.sqrt(2), (tension - pressure) / np.sqrt(2)
    return stack([first, second], axis=-1), stack([second, first], axis=-1)   # [..., 2] Point each


plane_normals, plane_slips = fault_planes(moment)                             # [faults, 2] Point each
rebuilt = double_couple(plane_normals, plane_slips, seismic_moment)           # [faults, 2] Point <- Point

## 5. Turning the rake

Keeping the fault and turning its slip once around, the ball goes from a normal fault, through strike-slip, to a thrust and back. In space, each direction is drawn as far as its P wave's first motion: four lobes, two pushing and two pulling, with the fault in between. This cell takes about twenty seconds, most of it drawing the lobes.

In [ ]:
# The fault whose rake turns: its strike and dip, in degrees, the frames of the turn, and the duration of each.
sweep_strike, sweep_dip = 30.0, 50.0
n_frames = 72
duration_ms = 80

In [ ]:
def sweep(n_frames: int):
    """Per frame of the rake's turn: the fault's normal, slip and principal axes, its P wave over the lower
    hemisphere, and the whole sphere of directions in the fault's frame with the P wave over it."""
    sphere = directions(np.linspace(0.0, np.pi, 2 * n_rings - 1), np.linspace(0.0, 2 * np.pi, n_spokes + 1))   # [2 * rings - 1, spokes + 1] Point
    for rake in np.linspace(-np.pi, np.pi, n_frames, endpoint=False):
        frame = fault_frame(np.deg2rad(sweep_strike), np.deg2rad(sweep_dip), np.array(rake))   # [] Rotor
        normal, slip = frame >> -down, frame >> north                         # [] Point each
        fault_moment = double_couple(normal, slip, seismic_moment)            # [] Point <- Point
        _, fault_axes = fault_moment.eigh()                                   # [3] Point
        # The sphere turned with the fault, so that its nodal planes lie along the grid lines.
        fault_sphere = frame >> sphere                                        # [2 * rings - 1, spokes + 1] Point
        ball, _ = radiation(fault_moment, hemisphere)                         # [rings, spokes + 1] Scalar
        lobes, _ = radiation(fault_moment, fault_sphere)                      # [2 * rings - 1, spokes + 1] Scalar
        yield normal, slip, fault_axes, ball, fault_sphere, lobes


show_animation(render.animate_sweep(hemisphere, sweep(n_frames)), "earthquake_rake", duration_ms)

In [ ]:
# checks
# A slip neither swells nor shrinks the ground and has no third axis.
np.testing.assert_allclose(moment.trace().to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(moment.det().to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(values.to_array(), np.array([-seismic_moment, 0.0, seismic_moment]) * np.ones((len(strikes), 1)), atol=1e-13)
# One couple turns the ground by the plane of shear; the double couple does not.
np.testing.assert_allclose(curl(single).kernel, slips.commutator(normals).kernel, atol=1e-14)
np.testing.assert_allclose(curl(moment).kernel, 0.0, atol=1e-14)
# The fault and the auxiliary plane give the same moment, and both planes found from it rebuild it.
np.testing.assert_allclose(auxiliary.kernel, moment.kernel, atol=1e-14)
np.testing.assert_allclose(rebuilt.kernel, moment[:, None].broadcast_to(rebuilt.shape).kernel, atol=1e-13)
np.testing.assert_allclose(plane_normals.dot(normals[:, None]).abs().to_array().max(axis=-1), 1.0, rtol=1e-12)
# Half a turn of rake reverses the slip and flips the moment.
sweep_frames = fault_frame(np.deg2rad(sweep_strike), np.deg2rad(sweep_dip), np.linspace(-np.pi, np.pi, n_frames, endpoint=False))   # [frames] Rotor
sweep_moment = double_couple(sweep_frames >> -down, sweep_frames >> north, seismic_moment)   # [frames] Point <- Point
np.testing.assert_allclose((sweep_moment[n_frames // 2:] + sweep_moment[:n_frames // 2]).kernel, 0.0, atol=1e-14)